# Step 4 — Emotion & Prosody Analysis (Emotion Radar)
**Tough Talks · Phase 2**

Goal: prove Gemma 4 E2B's **native audio path** can score a single
conversation turn at a time and emit JSON conforming to
`data/schemas/emotion_radar.schema.json`.

Same multimodal model and audio path as Step 03 — prosody, words, and
emotion all share one set of activations. Notebook is a thin driver;
all logic lives in `backend/core/_runtime/emotion.py`.

**Test audio — license note for Kaggle**
Same gTTS strategy as Step 03: per-turn argument clips synthesised
in-notebook (MIT-licensed). No third-party audio is downloaded, so
there is no upstream license to attribute.

**What "done" looks like for this step**
1. Multimodal Gemma 4 loads (`AutoModelForMultimodalLM` via `LoadConfig(multimodal=True)`).
2. Each per-turn clip is under Gemma 4's 30 s cap.
3. `analyze_emotion()` returns an `EmotionRadarResult` for each turn — `primary` in the schema enum, numerics in `[0, 1]`, `speaker`/`turn_id`/`timestamp` attached.
4. `analyze_emotion_long()` demonstrates the chunked path on the combined > 30 s clip, returning one reading per chunk window.
5. Every result validates against `data/schemas/emotion_radar.schema.json` (required fields, enum, numeric ranges).

**Gemma 4 audio constraints worth remembering** (unchanged from Step 03)
- Audio content must come **before** the text instruction in `content`.
- Per-call cap is **30 seconds** (`MAX_AUDIO_SECONDS`); longer clips are chunked at `DEFAULT_CHUNK_SECONDS` (28 s) with a small overlap.
- Audio support is on **E2B and E4B only**.

In [1]:
# ── 0. Install / upgrade dependencies ────────────────────────────────────────
# Same as Step 03 — only bump transformers + accelerate + audio libs. DO NOT
# bump torch on Colab/Kaggle (it breaks the pre-installed torchvision/CUDA
# pairing). After the first run, RESTART THE KERNEL before continuing —
# the already-imported transformers won't pick up the upgrade in place.

!pip install -q -U transformers accelerate librosa soundfile gTTS

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 85.1 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 6.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer-slim 0.24.0 requires typer>=0.24.0, but you have typer 0.23.1 which is incompatible.


In [2]:
# ── 1. Locate (or fetch) the repo, put it on sys.path ───────────────────────
# Same shim as Steps 01–03 — auto-clones / refreshes on Colab / Kaggle. After
# refreshing the working tree we also drop any cached `backend.*` modules
# from sys.modules so subsequent `from backend.core._runtime import X` picks
# up the freshly-pulled code instead of whatever this kernel imported earlier
# in the session.

import os, pathlib, subprocess, sys

REPO_URL  = "https://github.com/EhsanFarazmand/tough_talks.git"
REPO_NAME = "tough_talks"

def _looks_like_repo(p: pathlib.Path) -> bool:
    return (p / "backend" / "core" / "_runtime").is_dir()

def _scan_for_repo() -> pathlib.Path | None:
    cwd = pathlib.Path.cwd()
    for parent in [cwd, *cwd.parents]:
        if _looks_like_repo(parent):
            return parent
    for base in (pathlib.Path("/content"), pathlib.Path("/kaggle/working")):
        candidate = base / REPO_NAME
        if _looks_like_repo(candidate):
            return candidate
    return None

def _refresh(target: pathlib.Path) -> None:
    if not (target / ".git").is_dir():
        return
    print(f"Refreshing {target} from origin")
    subprocess.run(["git", "-C", str(target), "fetch", "--depth", "1", "origin"],
                   capture_output=True, check=False)
    subprocess.run(["git", "-C", str(target), "reset", "--hard", "FETCH_HEAD"],
                   capture_output=True, check=False)

REPO_ROOT = _scan_for_repo()
if REPO_ROOT is None:
    base = next((b for b in (pathlib.Path("/content"), pathlib.Path("/kaggle/working")) if b.is_dir()),
                pathlib.Path.cwd())
    target = base / REPO_NAME
    print(f"Cloning {REPO_URL} -> {target}")
    result = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(target)],
                            capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError("git clone failed:\n" + result.stderr)
    REPO_ROOT = target
else:
    _refresh(REPO_ROOT)

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

_stale = [m for m in list(sys.modules) if m == "backend" or m.startswith("backend.")]
for _m in _stale:
    del sys.modules[_m]
if _stale:
    print(f"Cleared {len(_stale)} cached backend.* module(s) from sys.modules")

print(f"Repo root: {REPO_ROOT}")

Cloning https://github.com/EhsanFarazmand/tough_talks.git -> /content/tough_talks
Repo root: /content/tough_talks


In [3]:
# ── 2. Imports ───────────────────────────────────────────────────────────────
import io
import json
from pathlib import Path

import numpy as np
import torch

from backend.core._runtime import (
    ALLOWED_EMOTIONS,
    ALLOWED_SPEAKERS,
    DEFAULT_CHUNK_SECONDS,
    DEFAULT_MODEL_ID,
    EmotionAnalysisError,
    EmotionConfig,
    JsonParseError,
    LoadConfig,
    MAX_AUDIO_SECONDS,
    analyze_emotion,
    analyze_emotion_long,
    format_emotion_context,
    load_model,
)

In [4]:
# ── 3. Configuration ─────────────────────────────────────────────────────────

MODEL_ID = DEFAULT_MODEL_ID                                # google/gemma-4-E2B-it
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"

# Same nine-turn argument script as Step 03, tagged with speaker labels so
# the Emotion Radar can frame `whisper_prompt` for the user only. The "user"
# speaks turns 1/3/5/7/9 (US accent), the "other" speaks 2/4/6/8 (UK accent).
# Speaker label drives the prompt; accent is purely cosmetic.
ARGUMENT_SCRIPT = [
    ("user",  "us", "I asked for the report on Monday and it's already Thursday. What happened?"),
    ("other", "uk", "I told you on Tuesday the data team hadn't delivered. I can't make numbers up."),
    ("user",  "us", "Then you escalate. You don't just sit on it. The whole quarter close depends on this."),
    ("other", "uk", "I did escalate. You weren't in the meeting. Don't blame me for your missed message."),
    ("user",  "us", "Look — I can't be in every meeting. That's why we have email."),
    ("other", "uk", "I sent two emails. You replied to neither. Don't put this on me."),
    ("user",  "us", "Okay, fair. I missed them. But we still have a problem to solve tonight."),
    ("other", "uk", "I have partial numbers from the staging tables. We can present those and flag the gaps."),
    ("user",  "us", "Good. Let's regroup at six. And next time, just call me directly."),
]

TARGET_SR    = 16000  # Gemma 4's audio extractor resamples to 16 kHz anyway
SILENCE_S    = 0.25
TLD_MAP      = {"us": "com", "uk": "co.uk"}

AUDIO_DIR    = Path(REPO_ROOT) / "data" / "audio_cache"
TURNS_DIR    = AUDIO_DIR / "turns"
COMBINED_WAV = AUDIO_DIR / "argument_synth.wav"

SCHEMA_PATH  = Path(REPO_ROOT) / "data" / "schemas" / "emotion_radar.schema.json"

print(f"Model       : {MODEL_ID}")
print(f"Device      : {DEVICE}")
print(f"Per-call cap: {MAX_AUDIO_SECONDS}s   chunk size: {DEFAULT_CHUNK_SECONDS}s")
print(f"Turn clips  : {TURNS_DIR}")
print(f"Combined    : {COMBINED_WAV}")
print(f"Speakers    : {ALLOWED_SPEAKERS}")
print(f"Emotion enum: {ALLOWED_EMOTIONS}")
print(f"Script      : {len(ARGUMENT_SCRIPT)} turns, ~{sum(len(t.split()) for _, _, t in ARGUMENT_SCRIPT)} words")

Model       : google/gemma-4-E2B-it
Device      : cuda
Per-call cap: 30s   chunk size: 28.0s
Turn clips  : /content/tough_talks/data/audio_cache/turns
Combined    : /content/tough_talks/data/audio_cache/argument_synth.wav
Speakers    : ('user', 'other')
Emotion enum: ('anger', 'fear', 'sadness', 'joy', 'surprise', 'disgust', 'neutral', 'frustration', 'openness', 'defensiveness')
Script      : 9 turns, ~127 words


In [5]:
# ── 4. Synthesise per-turn clips + combined argument WAV ─────────────────────
# Each turn becomes its own WAV in data/audio_cache/turns/. We also concat
# them into one long WAV (same file Step 03 produced) so the chunked path can
# be exercised in cell 7. All files are gitignored via the existing *.wav and
# audio_cache/ rules. Idempotent: skip whatever's already cached.

import librosa
import soundfile as sf
from gtts import gTTS

AUDIO_DIR.mkdir(parents=True, exist_ok=True)
TURNS_DIR.mkdir(parents=True, exist_ok=True)

turn_paths: list[Path] = []
combined_chunks: list[np.ndarray] = []
silence = np.zeros(int(TARGET_SR * SILENCE_S), dtype=np.float32)

for i, (speaker, accent, line) in enumerate(ARGUMENT_SCRIPT, start=1):
    turn_path = TURNS_DIR / f"turn_{i:02d}_{speaker}.wav"
    if not turn_path.exists():
        buf = io.BytesIO()
        gTTS(text=line, lang="en", tld=TLD_MAP[accent]).write_to_fp(buf)
        buf.seek(0)
        wave, _ = librosa.load(buf, sr=TARGET_SR, mono=True)
        sf.write(str(turn_path), wave.astype(np.float32), TARGET_SR)
    turn_paths.append(turn_path)
    wave, _ = librosa.load(str(turn_path), sr=TARGET_SR, mono=True)
    combined_chunks.append(wave.astype(np.float32))
    combined_chunks.append(silence)
    duration = sf.info(str(turn_path)).frames / TARGET_SR
    print(f"  turn {i:2d} [{speaker:5s}] {duration:5.2f}s — {line[:60]}{'...' if len(line) > 60 else ''}")

if not COMBINED_WAV.exists():
    sf.write(str(COMBINED_WAV), np.concatenate(combined_chunks), TARGET_SR)

combined_info = sf.info(str(COMBINED_WAV))
COMBINED_DURATION = combined_info.frames / combined_info.samplerate
print(f"\nCombined clip: {COMBINED_DURATION:.2f}s @ {combined_info.samplerate} Hz")
if COMBINED_DURATION > MAX_AUDIO_SECONDS:
    print(f"  → exceeds Gemma 4's {MAX_AUDIO_SECONDS}s cap — analyze_emotion_long() will chunk it.")
else:
    print(f"  → fits in a single window; analyze_emotion_long() will run one pass.")

  turn  1 [user ]  5.35s — I asked for the report on Monday and it's already Thursday. ...
  turn  2 [other]  5.52s — I told you on Tuesday the data team hadn't delivered. I can'...
  turn  3 [user ]  6.65s — Then you escalate. You don't just sit on it. The whole quart...
  turn  4 [other]  6.24s — I did escalate. You weren't in the meeting. Don't blame me f...
  turn  5 [user ]  4.97s — Look — I can't be in every meeting. That's why we have email...
  turn  6 [other]  6.12s — I sent two emails. You replied to neither. Don't put this on...
  turn  7 [user ]  5.88s — Okay, fair. I missed them. But we still have a problem to so...
  turn  8 [other]  6.24s — I have partial numbers from the staging tables. We can prese...
  turn  9 [user ]  5.93s — Good. Let's regroup at six. And next time, just call me dire...

Combined clip: 55.15s @ 16000 Hz
  → exceeds Gemma 4's 30s cap — analyze_emotion_long() will chunk it.


In [6]:
# ── 5. Load the multimodal processor + model ─────────────────────────────────
# Same shape as Step 03: multimodal=True so the audio inputs in
# apply_chat_template() are actually consumed by the model.

processor, model = load_model(LoadConfig(model_id=MODEL_ID, multimodal=True))
n_params = sum(p.numel() for p in model.parameters()) / 1e9
print(f"Model loaded ({n_params:.1f}B parameters, on {model.device})")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Model loaded (5.1B parameters, on cuda:0)


In [7]:
# ── 6. Per-turn emotion analysis (the realistic Live-Mode flow) ──────────────
# Each turn is well under the 30 s cap, so we use analyze_emotion() directly —
# no chunking needed. cfg.speaker shapes the prompt (whisper_prompt is forced
# to None in code for `other`, kept for `user`). cfg.prior_context threads a
# rolling summary of prior turns into the prompt so each whisper reflects the
# arc, not just the current turn — this is what makes Live-Mode coaching
# context-aware rather than turn-by-turn amnesiac.

turn_results: list[dict] = []
turn_errors: list[str] = []

for i, (turn_path, (speaker, _accent, _text)) in enumerate(zip(turn_paths, ARGUMENT_SCRIPT), start=1):
    cfg = EmotionConfig(
        speaker=speaker,
        turn_id=f"turn_{i:02d}",
        prior_context=format_emotion_context(turn_results),
    )
    try:
        result = analyze_emotion(processor, model, turn_path, cfg=cfg)
    except (EmotionAnalysisError, JsonParseError) as exc:
        turn_errors.append(f"turn {i}: {exc}")
        print(f"  turn {i:2d} [{speaker:5s}] FAILED: {exc}")
        continue
    turn_results.append(result)
    e = result["emotions"]
    coach = f" whisper={result['whisper_prompt']!r}" if result.get("whisper_prompt") else ""
    alert = f" ALERT={result['escalation_alert']!r}" if result.get("escalation_alert") else ""
    print(
        f"  turn {i:2d} [{speaker:5s}] {e['primary']:14s} "
        f"int={e['intensity']:.2f} ten={e['tension_level']:.2f} esc={e['escalation_risk']:.2f}"
        f"{coach}{alert}"
    )

print(f"\nPer-turn analysis: {len(turn_results)}/{len(ARGUMENT_SCRIPT)} succeeded")

  turn  1 [user ] frustration    int=0.70 ten=0.60 esc=0.50 whisper='Try to stay calm and focus on the resolution. Ask a specific question about the delay.'
  turn  2 [other] frustration    int=0.85 ten=0.80 esc=0.75 ALERT='This is escalating quickly. Take a breath and try to lower your volume to manage the tension.'
  turn  3 [user ] frustration    int=0.85 ten=0.85 esc=0.75 whisper='Acknowledge their stress first. Validate the importance of the quarter close before presenting your plan.' ALERT='Your tone is very high-stakes. Slow down and focus on solutions, not just the problem.'
  turn  4 [other] defensiveness  int=0.75 ten=0.85 esc=0.75 ALERT='The defensiveness is increasing tension. Acknowledge the frustration while maintaining a solution-oriented focus.'
  turn  5 [user ] frustration    int=0.70 ten=0.60 esc=0.50 whisper='Focus on the specific need for email communication rather than a general complaint.'
  turn  6 [other] frustration    int=0.70 ten=0.60 esc=0.50
  turn  7 [use

In [8]:
# ── 7. Long-clip chunked path (analyze_emotion_long) ─────────────────────────
# Multi-minute Live-Mode case — one long audio stream, no turn-boundary
# information available upfront. analyze_emotion_long() chunks the wave at
# DEFAULT_CHUNK_SECONDS (28 s) and emits one EmotionRadarResult per chunk.
# Chunks are processed SEQUENTIALLY WITH ROLLING CONTEXT: each chunk's prompt
# sees a summary of the previous chunks (primary emotion, tension, escalation,
# transcript snippet) via format_emotion_context() so whisper_prompt and
# escalation_alert reflect the arc, not just the local 28 s window.
# Speaker stays "user" for the whole long run because the chunked path has no
# diarisation — that's a future step. The point here is the chunking contract
# and the context-threaded coaching, not per-chunk speaker accuracy.

long_results = analyze_emotion_long(
    processor,
    model,
    COMBINED_WAV,
    cfg=EmotionConfig(speaker="user", turn_id="combined"),
)
print(f"analyze_emotion_long: produced {len(long_results)} chunk reading(s)")
for j, r in enumerate(long_results, start=1):
    e = r["emotions"]
    coach = f" whisper={r['whisper_prompt']!r}" if r.get("whisper_prompt") else ""
    alert = f" ALERT={r['escalation_alert']!r}" if r.get("escalation_alert") else ""
    print(
        f"  chunk {j:2d} [{r['speaker']:5s}] {e['primary']:14s} "
        f"int={e['intensity']:.2f} ten={e['tension_level']:.2f} esc={e['escalation_risk']:.2f}"
        f"{coach}{alert}"
    )

analyze_emotion_long: produced 2 chunk reading(s)
  chunk  1 [user ] anger          int=0.85 ten=0.90 esc=0.90 whisper="Focus on the impact of the delay rather than assigning blame. Try a 'what's next' approach." ALERT='This is highly escalated. Focus on problem-solving and finding solutions for the quarter close.'
  chunk  2 [user ] frustration    int=0.75 ten=0.60 esc=0.30 whisper='You acknowledged missing the emails, which is a good step. Focus now on the solution (partial numbers) and the agreed next steps.'


In [9]:
# ── 8. Schema validation + results table ─────────────────────────────────────
# Hand-rolled validator (no jsonschema dep). For each EmotionRadarResult:
#   - required top-level fields present (turn_id, speaker, emotions, timestamp)
#   - speaker in {user, other}
#   - emotions.primary in the schema enum
#   - intensity / tension_level / escalation_risk are numeric in [0, 1]

schema = json.loads(SCHEMA_PATH.read_text(encoding="utf-8"))
SCHEMA_REQUIRED  = schema["required"]
EMOTION_REQUIRED = schema["properties"]["emotions"]["required"]
EMOTION_ENUM     = schema["properties"]["emotions"]["properties"]["primary"]["enum"]
SPEAKER_ENUM     = schema["properties"]["speaker"]["enum"]


def _validate_emotion_result(result: dict) -> list[str]:
    errs: list[str] = []
    for k in SCHEMA_REQUIRED:
        if k not in result:
            errs.append(f"missing top-level key {k!r}")
    if result.get("speaker") not in SPEAKER_ENUM:
        errs.append(f"speaker {result.get('speaker')!r} not in {SPEAKER_ENUM}")
    e = result.get("emotions")
    if not isinstance(e, dict):
        errs.append("emotions is not an object")
        return errs
    for k in EMOTION_REQUIRED:
        if k not in e:
            errs.append(f"emotions.{k!r} missing")
    if e.get("primary") not in EMOTION_ENUM:
        errs.append(f"emotions.primary {e.get('primary')!r} not in enum")
    for nk in ("intensity", "tension_level", "escalation_risk"):
        v = e.get(nk)
        if v is None:
            continue
        if isinstance(v, bool) or not isinstance(v, (int, float)):
            errs.append(f"emotions.{nk} not numeric: {v!r}")
        elif not 0.0 <= float(v) <= 1.0:
            errs.append(f"emotions.{nk} out of range: {v}")
    return errs


all_results = turn_results + long_results
per_result_errors = [_validate_emotion_result(r) for r in all_results]
n_clean = sum(1 for errs in per_result_errors if not errs)

long_expected_chunks = 2 if COMBINED_DURATION > DEFAULT_CHUNK_SECONDS else 1
checks: list[tuple[str, bool, str]] = [
    ("multimodal_model_loaded",  True,                                       f"{n_params:.1f}B params on {model.device}"),
    ("per_turn_runs",            len(turn_results) == len(ARGUMENT_SCRIPT),   f"{len(turn_results)}/{len(ARGUMENT_SCRIPT)} turns"),
    ("no_runtime_errors",        not turn_errors,                            f"{len(turn_errors)} error(s)"),
    ("long_path_chunked",        len(long_results) >= long_expected_chunks,  f"{len(long_results)} reading(s) for {COMBINED_DURATION:.1f}s"),
    ("schema_valid_per_result",  n_clean == len(all_results),                f"{n_clean}/{len(all_results)} clean"),
    ("primary_emotions_in_enum", all(r["emotions"]["primary"] in EMOTION_ENUM for r in all_results),
                                                                              f"{len({r['emotions']['primary'] for r in all_results})} distinct labels"),
]

print("=" * 72)
print("STEP 4 RESULTS — Gemma 4 native emotion radar")
print("=" * 72)
all_ok = True
for name, ok, note in checks:
    icon = "PASS" if ok else "FAIL"
    print(f"[{icon}]  {name:28s}  {note}")
    if not ok:
        all_ok = False

print("\nEmotion arc (per-turn analysis):")
for r in turn_results:
    e = r["emotions"]
    flags = (" DEF" if e["defensive"] else "    ") + (" CON" if e["concession_made"] else "    ")
    print(f"  {r['turn_id']:14s} [{r['speaker']:5s}] {e['primary']:14s} "
          f"int={e['intensity']:.2f} esc={e['escalation_risk']:.2f}{flags}")

if any(per_result_errors):
    print("\nSchema errors:")
    for r, errs in zip(all_results, per_result_errors):
        if not errs:
            continue
        print(f"  {r.get('turn_id', '?')}:")
        for line in errs:
            print(f"    - {line}")

if turn_errors:
    print("\nRuntime errors (per turn):")
    for line in turn_errors:
        print(f"  - {line}")

print()
print("OVERALL:", "READY FOR STEP 5" if all_ok else "FIX FAILURES ABOVE")

STEP 4 RESULTS — Gemma 4 native emotion radar
[PASS]  multimodal_model_loaded       5.1B params on cuda:0
[PASS]  per_turn_runs                 9/9 turns
[PASS]  no_runtime_errors             0 error(s)
[PASS]  long_path_chunked             2 reading(s) for 55.1s
[PASS]  schema_valid_per_result       11/11 clean
[PASS]  primary_emotions_in_enum      4 distinct labels

Emotion arc (per-turn analysis):
  turn_01        [user ] frustration    int=0.70 esc=0.50        
  turn_02        [other] frustration    int=0.85 esc=0.75        
  turn_03        [user ] frustration    int=0.85 esc=0.75        
  turn_04        [other] defensiveness  int=0.75 esc=0.75 DEF    
  turn_05        [user ] frustration    int=0.70 esc=0.50        
  turn_06        [other] frustration    int=0.70 esc=0.50        
  turn_07        [user ] frustration    int=0.75 esc=0.50        
  turn_08        [other] defensiveness  int=0.75 esc=0.75 DEF    
  turn_09        [user ] neutral        int=0.30 esc=0.10        

O